[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/24_rope_solution.ipynb)

# ✅ Solution: rope

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax.numpy as jnp
def apply_rope(q_LK, k_LK):
    # L=seq, K=head dim (even)
    L, K = q_LK.shape[-2:]
    pos_L1 = jnp.arange(L)[:, None]
    freq_K2 = 1.0 / (10000.0 ** (jnp.arange(0, K, 2) / K))
    cos_LK2 = jnp.cos(pos_L1 * freq_K2)
    sin_LK2 = jnp.sin(pos_L1 * freq_K2)
    def rotate(x_LK):
        x_even = x_LK[..., 0::2] * cos_LK2 - x_LK[..., 1::2] * sin_LK2
        x_odd = x_LK[..., 0::2] * sin_LK2 + x_LK[..., 1::2] * cos_LK2
        return jnp.stack((x_even, x_odd), -1).reshape(x_LK.shape)
    return rotate(q_LK), rotate(k_LK)


In [ ]:
# Verify
print(apply_rope)


In [ ]:
from jax_judge import check
check("rope")
